# Movie Recommendation System (Content-Based)
Run the cells **top to bottom**. Make sure `data/tmdb_5000_movies.csv` and `data/tmdb_5000_credits.csv` exist.

## Step 1: Load and explore the data

In [1]:
import pandas as pd, numpy as np, ast, pickle

movies  = pd.read_csv('data/tmdb_5000_movies.csv')
credits = pd.read_csv('data/tmdb_5000_credits.csv')
print(movies.shape, credits.shape)
movies.head(2)

(4803, 20) (4803, 4)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500


In [2]:
credits.head(2)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


## Step 2: Merge and keep useful columns

In [3]:
movies = movies.merge(credits, on='title')
movies = movies.rename(columns={'id': 'movie_id'})
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]
movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   movie_id  4809 non-null   int64
 1   movie_id  4809 non-null   int64
 2   title     4809 non-null   str  
 3   overview  4806 non-null   str  
 4   genres    4809 non-null   str  
 5   keywords  4809 non-null   str  
 6   cast      4809 non-null   str  
 7   crew      4809 non-null   str  
dtypes: int64(2), str(6)
memory usage: 300.7 KB


## Step 3: Clean (missing values and duplicates)

In [4]:
print(movies.isnull().sum())
movies.dropna(inplace=True)
movies.drop_duplicates(subset='title', inplace=True)
movies.reset_index(drop=True, inplace=True)
print(movies.shape)

movie_id    0
movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64
(4797, 8)


C:\Users\akshi\AppData\Local\Temp\ipykernel_20524\3093932837.py:2: Pandas4Warning: The inplace keyword in DataFrame.dropna is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  movies.dropna(inplace=True)
C:\Users\akshi\AppData\Local\Temp\ipykernel_20524\3093932837.py:3: Pandas4Warning: The inplace keyword in DataFrame.drop_duplicates is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  movies.drop_duplicates(subset='title', inplace=True)
C:\Users\akshi\AppData\Local\Temp\ipykernel_20524\3093932837.py:4: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  movies.reset_index(drop=True, inplace=True)


## Step 4: Convert the JSON-like text columns into lists

In [5]:
def get_names(text):
    return [d['name'] for d in ast.literal_eval(text)]

def get_top3_cast(text):
    return [d['name'] for d in ast.literal_eval(text)[:3]]

def get_director(text):
    return [d['name'] for d in ast.literal_eval(text) if d['job'] == 'Director']

movies['genres']   = movies['genres'].apply(get_names)
movies['keywords'] = movies['keywords'].apply(get_names)
movies['cast']     = movies['cast'].apply(get_top3_cast)
movies['crew']     = movies['crew'].apply(get_director)
movies['overview'] = movies['overview'].apply(lambda x: x.split())

# 'Sam Worthington' -> 'SamWorthington' so each person is a single token
for col in ['genres','keywords','cast','crew']:
    movies[col] = movies[col].apply(lambda lst: [i.replace(' ', '') for i in lst])

movies[['title','genres','cast','crew']].head()

,title,genres,cast,crew
0,Avatar,"[Action, Adventure, Fantasy, ScienceFiction]","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron]
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski]
2,Spectre,"[Action, Adventure, Crime]","[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes]
3,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[ChristianBale, MichaelCaine, GaryOldman]",[ChristopherNolan]
4,John Carter,"[Action, Adventure, ScienceFiction]","[TaylorKitsch, LynnCollins, SamanthaMorton]",[AndrewStanton]


## Step 5: Create the `tags` column

In [6]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']
movies['tags'] = movies['tags'].apply(lambda x: ' '.join(x).lower())
movies[['title','tags']].head(2)

,title,tags
0,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."


## Step 6: Stemming (NLTK)
Reduces words to their root so 'loving', 'loved' and 'loves' all match.

In [7]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()
movies['tags'] = movies['tags'].apply(lambda t: ' '.join(ps.stem(w) for w in t.split()))

## Step 7: Vectorize with CountVectorizer

In [8]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(movies['tags']).toarray()
print(vectors.shape)
cv.get_feature_names_out()[:20]

(4797, 5000)


array(['000', '007', '10', '100', '11', '12', '13', '14', '15', '16',
       '17', '17th', '18', '18th', '18thcenturi', '19', '1910', '1920',
       '1930', '1940'], dtype=object)

## Step 8: Cosine similarity

In [9]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vectors).astype('float32')  # float32 keeps the file under GitHub's 100 MB limit
similarity.shape

(4797, 4797)

## Step 9: Recommendation function and test

In [10]:
def recommend(title, n=5):
    if title not in movies['title'].values:
        return 'Movie not found'
    idx = movies[movies['title'] == title].index[0]
    scores = sorted(enumerate(similarity[idx]), key=lambda x: x[1], reverse=True)[1:n+1]
    return [movies.iloc[i].title for i, _ in scores]

print(recommend('Avatar'))
print(recommend('The Dark Knight'))

['Aliens vs Predator: Requiem', 'Aliens', 'Falcon Rising', 'Independence Day', 'Titan A.E.']
['The Dark Knight Rises', 'Batman Begins', 'Batman Returns', 'Batman Forever', 'Batman']


## Step 10: Save the model

In [11]:
pickle.dump(movies[['movie_id','title']], open('model.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))
print('Saved!')

Saved!


## Step 11 (Bonus): Evaluate quickly
A simple sanity check: recommendations for a movie should share genres with it.

In [12]:
raw = pd.read_csv('data/tmdb_5000_movies.csv')
raw['genre_set'] = raw['genres'].apply(lambda t: set(d['name'] for d in ast.literal_eval(t)))
g = dict(zip(raw['title'], raw['genre_set']))

def genre_overlap(title, n=5):
    recs = recommend(title, n)
    base = g.get(title, set())
    shared = [len(base & g.get(r, set())) > 0 for r in recs]
    return sum(shared) / len(shared)

sample = movies['title'].sample(min(50, len(movies)), random_state=1)
print('Average share of recommendations with a matching genre:', np.mean([genre_overlap(t) for t in sample]))

Average share of recommendations with a matching genre: 0.8439999999999999


## Bonus ideas
- **Collaborative filtering:** use MovieLens ratings with the `surprise` library (`SVD`, `KNNBasic`).
- **Hybrid:** `final = 0.5 * content_score + 0.5 * collaborative_score` (normalise both to 0-1).